In [1]:
import os
import io
import copy
import random
from tqdm.auto import tqdm 

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from torchvision import datasets, transforms, models
from torchvision.models import resnet18, ResNet18_Weights

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [2]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.247, 0.243, 0.261])
])

train_dataset_full = datasets.CIFAR10(root='./cifar-10-batches-py', train=True,
                                      download=True, transform=transform)
test_dataset_full = datasets.CIFAR10(root='./cifar-10-batches-py', train=False,
                                     download=True, transform=transform)

train_dataset = train_dataset_full
test_dataset = test_dataset_full

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("train - test dataset ready.")

Files already downloaded and verified
Files already downloaded and verified
train - test dataset ready.


In [3]:
def load_model(path="full_model.pth"):
    print("\nLoading pretrained ResNet18 model...")
    model = resnet18(weights=ResNet18_Weights.DEFAULT)

    # Modify final layer for 2 classes
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 10)

    # Freeze all layers except layer3, layer4, fc
    for param in model.parameters():
        param.requires_grad = False
    for layer in [model.layer3, model.layer4, model.fc]:
        for param in layer.parameters():
            param.requires_grad = True

    # Load fine-tuned weights if available
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device))
        print(f"Loaded fine-tuned weights from '{path}'")
    else:
        print(f"Warning: '{path}' not found. Model is not fine-tuned yet.")

    return model.to(device)


model = load_model()
model.eval()



Loading pretrained ResNet18 model...


/tmp/ipykernel_6668/1370779204.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path, map_location=device))


Loaded fine-tuned weights from 'full_model.pth'


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [4]:
class Encoder(nn.Module):
    def __init__(self, trained_model):
        super().__init__()
        self.features = nn.Sequential(
            trained_model.conv1,
            trained_model.bn1,
            trained_model.relu,
            trained_model.maxpool,
            trained_model.layer1,
            trained_model.layer2,
            trained_model.layer3
        )

    def forward(self, x):
        with torch.no_grad():
            return self.features(x)


encoder = Encoder(model).to(device)
encoder.eval()


Encoder(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True

In [5]:
class MirrorBasicBlock(nn.Module):
    """Reverse of a ResNet BasicBlock, preserving residual skip."""
    def __init__(self, block):
        super().__init__()
        # Mirror convs
        self.deconv2 = self.mirror_conv(block.conv2)
        self.bn2 = nn.BatchNorm2d(block.conv2.in_channels)
        self.deconv1 = self.mirror_conv(block.conv1)
        self.bn1 = nn.BatchNorm2d(block.conv1.in_channels)
        self.relu = nn.ReLU(inplace=True)

        # Mirror downsample (if present)
        if block.downsample is not None:
            down = block.downsample[0]
            self.upsample = nn.Sequential(
                self.mirror_conv(down),
                nn.BatchNorm2d(down.in_channels)
            )
        else:
            self.upsample = None

    def mirror_conv(self, conv):
        """Create a ConvTranspose2d layer that mirrors a given Conv2d layer."""
        return nn.ConvTranspose2d(
            in_channels=conv.out_channels,
            out_channels=conv.in_channels,
            kernel_size=conv.kernel_size,
            stride=conv.stride,
            padding=conv.padding,
            output_padding=1 if conv.stride[0] > 1 else 0,
            bias=False,
        )

    def forward(self, x):
        identity = x
        out = self.deconv2(x)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.deconv1(out)
        out = self.bn1(out)

        if self.upsample is not None:
            identity = self.upsample(identity)

        out += identity
        out = self.relu(out)
        return out


In [6]:
class Decoder(nn.Module):
    def __init__(self):
        super(Decoder, self).__init__()

        base = models.resnet18(weights=None)

        def mirror_layer(layer):
            blocks = [MirrorBasicBlock(b) for b in reversed(layer)]
            return nn.Sequential(*blocks)

        self.up3 = mirror_layer(base.layer3)
        self.up2 = mirror_layer(base.layer2)
        self.up1 = mirror_layer(base.layer1)

        # Mirror of initial conv (7x7 stride=2)
        self.final = nn.ConvTranspose2d(
            64, 3, kernel_size=7, stride=2, padding=3, output_padding=1
        )

        # Extra step: bring from 16×16 -> 32×32
        self.upsample_final = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(3, 3, kernel_size=3, stride=1, padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        x = self.up3(x)
        x = self.up2(x)
        x = self.up1(x)
        x = self.final(x)
        x = self.upsample_final(x)
        return x

decoder = Decoder().to(device)
model = model.to(device)

In [7]:
def set_seed(seed):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    np.random.seed(seed); random.seed(seed)
    torch.backends.cudnn.deterministic = True

seeds = [2, 19, 43, 127, 409]

# Directories
save_dir = "MRTanh_TargetAlpha"
os.makedirs(save_dir, exist_ok=True)

# Summary DF
summary_df = pd.DataFrame(columns=["alpha", "seed", "raw_CE", "raw_MSE", "mean_CE", "mean_MSE", "norm_CE", "norm_MSE", "L_Total"])
csv_path = os.path.join(save_dir, "training_summary.csv")

# Settings
criterion_ce = nn.CrossEntropyLoss()
criterion_mse = nn.MSELoss()
LR = 1e-4
NUM_EPOCHS = 1000
PATIENCE = 20
momentum = 0.9
epsilon = 1e-10
alphas = [0.18, 0.24, 0.27]

# Freeze encoder & classifier
for p in encoder.parameters(): p.requires_grad = False
for p in model.parameters(): p.requires_grad = False
encoder.eval(); model.eval()

# Train for each seed
for seed in seeds:
    set_seed(seed)
    print(f"\n=== Seed {seed} ===\n")

    for alpha in alphas:
        save_path = os.path.join(save_dir, f"decoder{alpha:.2f}_seed{seed}.pth")
        if os.path.exists(save_path):
            print(f"Skipping alpha {alpha:.2f}, seed {seed}")
            continue

        print(f"\nTraining alpha={alpha:.2f}, seed={seed}\n")
        decoder_curr = Decoder().to(device)
        decoder_curr.train()
        optimizer = optim.Adam(decoder_curr.parameters(), lr=LR)

        mean_ce, mean_mse = 1.0, 1.0
        best_loss = float("inf")
        best_state = None
        patience_counter = 0

        for epoch in range(NUM_EPOCHS):
            running_ce, running_mse, total = 0.0, 0.0, 0

            for images, labels in tqdm(train_loader, desc=f"A {alpha:.2f} S {seed} E {epoch+1}", leave=False):
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()

                latent = encoder(images)
                recon = decoder_curr(latent)
                logits = model(recon)

                L_CE_raw = criterion_ce(logits, labels)
                L_MSE_raw = criterion_mse(recon, images)

                L_CE_norm = L_CE_raw / (mean_ce + epsilon)
                L_MSE_norm = L_MSE_raw / (mean_mse + epsilon)
                L_Total = L_CE_norm + alpha * L_MSE_norm

                L_Total.backward()
                optimizer.step()

                b = images.size(0)
                running_ce += L_CE_raw.item() * b
                running_mse += L_MSE_raw.item() * b
                total += b

            epoch_ce = running_ce / total
            epoch_mse = running_mse / total
            epoch_ce_norm = epoch_ce / (mean_ce + epsilon)
            epoch_mse_norm = epoch_mse / (mean_mse + epsilon)
            epoch_total_norm = epoch_ce_norm + alpha * epoch_mse_norm

            mean_ce = momentum * mean_ce + (1 - momentum) * epoch_ce
            mean_mse = momentum * mean_mse + (1 - momentum) * epoch_mse

            if epoch_total_norm < best_loss - epsilon:
                best_loss = epoch_total_norm
                best_state = decoder_curr.state_dict()
                patience_counter = 0
                best_epoch_ce = epoch_ce
                best_epoch_mse = epoch_mse
                best_mean_ce = mean_ce
                best_mean_mse = mean_mse
                best_ce_norm = epoch_ce_norm
                best_mse_norm = epoch_mse_norm
                best_total_norm = epoch_total_norm
            else:
                patience_counter += 1

            print(f"E{epoch+1:03d} | CE={epoch_ce_norm:.6f} | MSE={epoch_mse_norm:.6f} | Total={epoch_total_norm:.6f} | P={patience_counter}")

            if patience_counter >= PATIENCE:
                print(f"Early stop at epoch {epoch+1}")
                break

        torch.save(best_state, save_path)
        print(f"Saved decoder alpha={alpha:.2f}, seed={seed}")

        summary_df = pd.concat([
            summary_df,
            pd.DataFrame([{
                "alpha": alpha,
                "seed": seed,
                "raw_CE": best_epoch_ce,
                "raw_MSE": best_epoch_mse,
                "mean_CE": best_mean_ce,
                "mean_MSE": best_mean_mse,
                "norm_CE": best_ce_norm,
                "norm_MSE": best_mse_norm,
                "L_Total": best_total_norm
            }])
        ], ignore_index=True)

summary_df.to_csv(csv_path, index=False)
print(f"\nSummary saved at {csv_path}")



=== Seed 2 ===


Training alpha=0.18, seed=2



A 0.18 S 2 E 1:   0%|          | 0/782 [00:00<?, ?it/s]

E001 | CE=0.700063 | MSE=1.293375 | Total=0.932871 | P=0


A 0.18 S 2 E 2:   0%|          | 0/782 [00:00<?, ?it/s]

E002 | CE=0.423511 | MSE=1.216748 | Total=0.642525 | P=0


A 0.18 S 2 E 3:   0%|          | 0/782 [00:00<?, ?it/s]

... (1,116 outputs left out)

A 0.27 S 409 E 40:   0%|          | 0/782 [00:00<?, ?it/s]

E040 | CE=0.379426 | MSE=1.005331 | Total=0.650866 | P=19


A 0.27 S 409 E 41:   0%|          | 0/782 [00:00<?, ?it/s]

E041 | CE=0.389853 | MSE=1.005293 | Total=0.661283 | P=20
Early stop at epoch 41
Saved decoder alpha=0.27, seed=409

Summary saved at MRTanh_TargetAlpha/training_summary.csv
